In [1]:
from getpass import getpass
import sys, os

REPO_NAME = "RecSys-Challenge-2025"
REPO_URL  = f"github.com/Lv1g1/{REPO_NAME}.git"
LOCAL_REPO_PATH = f"/content/{REPO_NAME}"

# Mount Google Drive first (Colab only)
if '/content' in os.getcwd():
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

# Clone only if not existing
if not os.path.exists(LOCAL_REPO_PATH):
    print("Cloning repo...")
    token = getpass("GitHub Token: ")
    !git clone https://{token}@{REPO_URL}
else:
    print("Repo already exists — pulling latest changes")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
    os.chdir("/content")

# Enable import of your modules
if LOCAL_REPO_PATH not in sys.path:
    os.chdir(LOCAL_REPO_PATH)
    sys.path.append(os.getcwd())
    os.chdir("/content")

Mounted at /content/drive
Cloning repo...
GitHub Token: ··········
Cloning into 'RecSys-Challenge-2025'...
remote: Enumerating objects: 225, done.
remote: Counting objects: 100% (225/225), done.
remote: Compressing objects: 100% (172/172), done.
remote: Total 225 (delta 67), reused 202 (delta 47), pack-reused 0 (from 0)
Receiving objects: 100% (225/225), 7.34 MiB | 23.19 MiB/s, done.
Resolving deltas: 100% (67/67), done.


In [3]:
!pip install optuna
import optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.9/400.9 kB 8.8 MB/s eta 0:00:00


In [ ]:
import importlib
import scipy.sparse as sps
import json

from Challenge import paths
importlib.reload(paths)

from Evaluation.Evaluator import EvaluatorHoldout
from Challenge.hyper_tuning import hyperparameter_tuning

Running on: colab — BASE_DIR = /content/drive/MyDrive/RecSys/demo


In [5]:
# Load datasets
URM_train = sps.load_npz(paths.URM_TRAIN)
URM_validation = sps.load_npz(paths.URM_VALIDATION)

In [6]:
# Set up evaluator
evaluator = EvaluatorHoldout(URM_validation, cutoff_list=[10])

EvaluatorHoldout: Ignoring 76 ( 0.1%) Users that have less than 1 test interactions


In [ ]:
# Define objective function for hyperparameter tuning
from Recommenders.NonPersonalizedRecommender import GlobalEffects

STUDY_NAME = GlobalEffects.RECOMMENDER_NAME + "_hyperparameter_tuning"

def objective_function(optuna_trial: optuna.trial.Trial) -> float:
    recommender_instance = GlobalEffects(URM_train)
    recommender_instance.fit(
        # shrink factor
        lambda_user=optuna_trial.suggest_int("lambda_user", 0, 1000),
        lambda_item=optuna_trial.suggest_int("lambda_item", 0, 1000)
    )

    result_df, _ = evaluator.evaluateRecommender(recommender_instance)

    return result_df.loc[10]["MAP"]

In [ ]:
# Perform hyperparameter tuning
save_results, optuna_study = hyperparameter_tuning(
    objective_function,
    study_name=STUDY_NAME,
    n_trials=50
)

In [ ]:
# Visualize optimization history and parameter importances
optuna.visualization.plot_optimization_history(optuna_study)
optuna.visualization.plot_param_importances(optuna_study)
optuna.visualization.plot_parallel_coordinate(optuna_study)

In [ ]:
# Train final model on train + validation with best hyperparameters
recommender = GlobalEffects(URM_train + URM_validation)
recommender.fit(
    lambda_user=optuna_study.best_trial.params["lambda_user"],
    lambda_item=optuna_study.best_trial.params["lambda_item"]
)

# Save the trained model
recommender.save_model(paths.MODEL_DIR)